# Day 19. GPT 구조

**오늘의 목표**
- GPT가 Transformer의 어떤 부분을 쓰는지 (Decoder-only) 설명할 수 있다
- 지금까지 만든 부품(Embedding, Position, Block, LayerNorm, Linear)을 **하나의 `GPT` 모델**로 조립한다
- 입력 Token ID `(B, T)`가 출력 logits `(B, T, vocab_size)`가 될 때까지 **모든 shape를 따라간다**
- 학습 전 GPT의 출력이 왜 "거의 균등한 확률"이고, Loss가 왜 `log(vocab_size)` 근처인지 안다
- GPT-2 small이 왜 약 1.24억 개 파라미터인지 직접 세어본다

> 오늘은 **조립과 forward 한 번**까지입니다. 학습(training)과 문장 생성 루프는 Day 20과 4주차에서 합니다.
>
> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요. `# TODO` 셀은 연습 문제입니다.

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

def count(module):
    # 파라미터 개수 (weight tying으로 공유된 텐서는 한 번만 셈)
    return sum(p.numel() for p in module.parameters())

## 1. GPT는 Transformer의 어느 부분인가?

원본 Transformer(2017)는 번역용이라 **Encoder + Decoder** 두 덩어리였습니다. (Day 18)

```text
원본 Transformer                       GPT
─────────────────                      ─────────────────
Encoder (원문 읽기)                     (없음)
Decoder                                Decoder
  ├ Masked Self-Attention                ├ Masked Self-Attention  ✅
  ├ Cross-Attention (Encoder 참고)        ├ (없음: 참고할 Encoder가 없으니까)
  └ FFN                                  └ FFN                    ✅
```

GPT는 **Decoder에서 Cross-Attention을 뺀 Block만** 여러 층 쌓은 모델입니다. 그래서 **Decoder-only** 라고 부릅니다.
할 일이 "앞 Token들을 보고 다음 Token 맞히기" 하나뿐이라 Encoder가 필요 없어요.

커리큘럼 5-2의 그림을 오늘 코드로 그대로 옮깁니다.

```text
Token
 ↓
Embedding            ← wte (Token) + wpe (Position)      Day 9, 10
 ↓
Transformer Block    ← LayerNorm → Attention → +         Day 14~17
 ↓                     LayerNorm → FFN → +
Transformer Block
 ↓
...  (n_layer 번)
 ↓
LayerNorm            ← ln_f (마지막 정리)                  오늘
 ↓
Linear               ← lm_head: C → vocab_size           Day 3, 9
 ↓
Softmax              ← 확률                               Day 5
 ↓
Next Token
```

## 2. 부품 준비 (Day 14~17에서 만든 것)

새로 배우는 내용은 없습니다. 오늘 노트북이 혼자 돌아가도록 그대로 가져온 코드예요.
- `CausalSelfAttention`: Day 14 완성본 그대로
- `MLP`: Day 17의 FFN (`Linear(C, 4C) → GELU → Linear(4C, C)`)
- `Block`: Day 15~16의 **Pre-LN** 구조 (`x + Attention(LN(x))`, `x + MLP(LN(x))`)

In [ ]:
# ===== Day 15~17에서 만든 것 =====

class CausalSelfAttention(nn.Module):           # Day 14 완성본

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd는 n_head로 나누어떨어져야 합니다"
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)   # Q, K, V 한 번에
        self.proj   = nn.Linear(n_embd, n_embd)                   # Head 합친 뒤 정리
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        hs = C // self.n_head

        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)        # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * hs ** -0.5             # (B, nh, T, T)
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v                                              # (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)         # (B, T, C)
        return self.proj(y)


class MLP(nn.Module):                            # Day 17 FFN

    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):                          # Day 15~16 Pre-LN Block

    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln2  = nn.LayerNorm(n_embd)
        self.mlp  = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))           # Residual ①
        x = x + self.mlp(self.ln2(x))            # Residual ②
        return x

print("부품 준비 완료: CausalSelfAttention, MLP, Block")

## 3. 설정값 묶기: `GPTConfig`

GPT 크기를 정하는 숫자는 딱 6개입니다. 한곳에 모아두면 작은 모델 ↔ GPT-2 크기를 숫자만 바꿔서 만들 수 있어요.

| 이름 | 뜻 | 오늘 작은 모델 | GPT-2 small |
|---|---|---|---|
| `vocab_size` | Token 종류 수 | 65 | 50257 |
| `block_size` | 한 번에 볼 수 있는 최대 Token 수 (context length) | 8 | 1024 |
| `n_layer` | Block 개수 | 2 | 12 |
| `n_head` | Head 개수 | 4 | 12 |
| `n_embd` | Token 벡터 길이 `C` | 32 | 768 |
| `dropout` | 학습 때 무작위로 끄는 비율 | 0.0 | 0.1 |

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 8
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 32
    dropout: float = 0.0

cfg = GPTConfig()
print(cfg)

## 4. `GPT` 클래스 조립하기

부품은 5개입니다.

| 이름 | 정체 | shape 변화 | 배운 날 |
|---|---|---|---|
| `wte` | Token Embedding (`nn.Embedding(vocab_size, C)`) | `(B, T)` → `(B, T, C)` | Day 9 |
| `wpe` | Position Embedding (`nn.Embedding(block_size, C)`) | `(T,)` → `(T, C)` | Day 10 |
| `blocks` | Block × `n_layer` | `(B, T, C)` → `(B, T, C)` | Day 14~17 |
| `ln_f` | 마지막 LayerNorm | `(B, T, C)` → `(B, T, C)` | Day 16 |
| `lm_head` | Linear `C → vocab_size` (bias 없음) | `(B, T, C)` → `(B, T, vocab_size)` | Day 3 |

### 4-1. 왜 마지막에 `ln_f`가 또 있나?

Pre-LN Block은 `x + Attention(LN(x))`처럼 **LayerNorm을 거친 값은 Attention에만 들어가고, Residual 줄기 `x` 자체는 정규화되지 않은 채** 다음 Block으로 넘어갑니다.
Block을 여러 개 지나면 줄기의 값이 계속 더해져서 크기가 커져요.
그래서 `lm_head`에 넣기 직전에 **한 번 더 정리**해 줍니다. (Day 16의 Post-LN 구조였다면 Block 끝에 이미 LN이 있어서 필요 없었어요)

### 4-2. Weight Tying (Day 9 복습)

`wte.weight`는 `(vocab_size, C)`, `lm_head.weight`도 `(vocab_size, C)`로 shape가 같습니다.
- `wte`: Token ID → 벡터 (**읽기**)
- `lm_head`: 벡터 → 각 Token 점수 (**쓰기**: 각 Token 벡터와 Dot Product)

같은 "Token 사전"을 양쪽에서 쓰자는 아이디어라 GPT-2는 **같은 텐서를 공유**합니다. 파라미터도 크게 줄어요.

### 4-3. 초기값을 작게 (std 0.02)

PyTorch 기본 초기값을 그대로 쓰면 학습 전부터 logits가 크게 튀어서 Loss가 이상하게 높게 시작합니다. (6-2에서 직접 확인)
GPT-2는 모든 Linear·Embedding을 **평균 0, 표준편차 0.02**로 초기화해요.

In [ ]:
class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config
        C = config.n_embd

        self.wte = nn.Embedding(config.vocab_size, C)          # Token Embedding
        self.wpe = nn.Embedding(config.block_size, C)          # Position Embedding
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([
            Block(C, config.n_head, config.block_size, config.dropout)
            for _ in range(config.n_layer)
        ])
        self.ln_f = nn.LayerNorm(C)                            # 마지막 LayerNorm
        self.lm_head = nn.Linear(C, config.vocab_size, bias=False)

        self.lm_head.weight = self.wte.weight                  # Weight Tying

        self.apply(self._init_weights)                         # 모든 부품에 초기화 적용

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if getattr(module, "bias", None) is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx):
        B, T = idx.shape
        assert T <= self.config.block_size, f"입력 길이 {T}가 block_size {self.config.block_size}보다 깁니다"

        pos = torch.arange(T, device=idx.device)               # (T,)  [0, 1, ..., T-1]
        x = self.wte(idx) + self.wpe(pos)                      # (B, T, C) + (T, C) → (B, T, C)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)                                       # (B, T, C)
        x = self.ln_f(x)                                       # (B, T, C)
        logits = self.lm_head(x)                               # (B, T, vocab_size)
        return logits

model = GPT(cfg)
print("파라미터 개수:", f"{count(model):,}")
print("Weight Tying 확인:", model.lm_head.weight is model.wte.weight)

> 💡 `forward`는 겨우 10줄입니다. GPT의 "구조"는 이게 전부예요.
> 나머지 복잡함은 전부 `Block` 안(Attention, FFN)에 숨어 있고, 그건 이미 2주차~3주차에 만들었습니다.

## 5. 문장 하나를 넣고 shape 따라가기

Day 8의 글자 단위 Tokenizer로 `"나는 오늘 학교에 간다"`를 Token ID로 바꿉니다.

In [ ]:
text = "나는 오늘 학교에 간다"
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

ids = encode(text)
print("글자 종류:", chars, f"({len(chars)}개)")
print("Token ID :", ids)
print("Token 수 :", len(ids))

> `vocab_size = 65`인데 지금 쓰는 글자는 10개뿐이에요. 65는 4주차에 쓸 영어 텍스트(셰익스피어) 글자 수에 맞춘 값입니다.
> 지금은 ID 0~9만 쓰고 나머지 55개 칸은 비어 있다고 생각하면 됩니다.

### 5-1. `block_size`보다 길면?

문장은 Token 12개인데 `block_size = 8`입니다. 그대로 넣으면 어떻게 될까요?

In [ ]:
idx_full = torch.tensor([ids])          # (1, 12)
try:
    model(idx_full)
except AssertionError as e:
    print("AssertionError:", e)

Position Embedding 표(`wpe`)에 0~7번 위치만 있어서 8번 이후 위치는 표현할 수가 없습니다. (Day 10)
이게 바로 **context length 제한**이에요. 긴 글을 다룰 때는 **마지막 `block_size`개만 잘라서** 넣습니다.

In [ ]:
idx = idx_full[:, -cfg.block_size:]     # 마지막 8개만
print("잘라낸 입력:", repr(decode(idx[0].tolist())))
print("idx shape  :", tuple(idx.shape))

### 5-2. forward 안을 한 줄씩 따라가기

`model(idx)`가 안에서 하는 일을 직접 풀어서 실행하며 shape를 찍어봅니다. (학습 원칙 3: 직접 값을 출력한다)

In [ ]:
B, T = idx.shape
print(f"{'입력 idx':22s} {tuple(idx.shape)}")

tok = model.wte(idx)
print(f"{'① wte(idx)':22s} {tuple(tok.shape)}")

pos = model.wpe(torch.arange(T))
print(f"{'② wpe(arange(T))':22s} {tuple(pos.shape)}")

x = tok + pos
print(f"{'③ tok + pos':22s} {tuple(x.shape)}   ← (B,T,C) + (T,C) broadcasting")

for i, block in enumerate(model.blocks):
    x = block(x)
    print(f"{'④ Block ' + str(i):22s} {tuple(x.shape)}")

x = model.ln_f(x)
print(f"{'⑤ ln_f':22s} {tuple(x.shape)}")

logits = model.lm_head(x)
print(f"{'⑥ lm_head':22s} {tuple(logits.shape)}   ← 위치마다 vocab_size개 점수")

print()
print("model(idx)와 같은가?", torch.allclose(logits, model(idx)))

**한 줄 요약**

```text
(B, T)  →  (B, T, C)  →  Block × n_layer  →  (B, T, C)  →  (B, T, vocab_size)
 ID         벡터           shape 그대로         정리          점수
```

`(1, 8, 65)`의 뜻: 8개 위치 **각각**에서 "다음 글자가 65개 중 무엇일지" 점수를 냅니다.
- 위치 0의 점수 = `"오"`만 보고 다음 글자 예측
- 위치 7의 점수 = `"오늘 학교에 간다"` 전체를 보고 다음 글자 예측

Causal Mask(Day 14) 덕분에 각 위치는 자기보다 앞만 봅니다. 이 "위치마다 다음 Token 맞히기"를 학습시키는 게 Day 20입니다.

## 6. 학습 전 GPT는 무엇을 출력할까?

### 6-1. 확률은 거의 균등하다

In [ ]:
probs = F.softmax(logits, dim=-1)        # (1, 8, 65)
last = probs[0, -1]                      # 마지막 위치의 다음 글자 확률

print("균등 확률 1/65   :", round(1 / cfg.vocab_size, 4))
print("가장 큰 확률     :", round(last.max().item(), 4))
print("가장 작은 확률   :", round(last.min().item(), 4))

plt.figure(figsize=(10, 3))
plt.bar(range(cfg.vocab_size), last.detach(), color="#4C72B0")
plt.axhline(1 / cfg.vocab_size, color="red", linestyle="--", label="균등 확률 1/65")
plt.title("학습 전 GPT: 마지막 위치의 다음 글자 확률", fontsize=13)
plt.xlabel("Token ID")
plt.ylabel("확률")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

65개 글자에 확률이 거의 똑같이 퍼져 있습니다. 아직 아무것도 배우지 않았으니 **"모르겠다, 다 비슷하다"** 상태인 게 정상이에요.

### 6-2. Loss는 `log(vocab_size)` 근처 (Day 6 복습)

정답 Token을 아무거나 정해서 Cross Entropy를 재봅니다. (진짜 정답을 만드는 방법은 Day 20)

In [ ]:
targets = torch.randint(0, cfg.vocab_size, (B, T))
loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))   # Day 6의 reshape

print("학습 전 Loss       :", round(loss.item(), 4))
print("log(vocab_size)    :", round(math.log(cfg.vocab_size), 4))

4주차에 mini GPT 학습을 시작하면 **첫 Loss가 약 4.17**로 찍힐 거예요. 그 숫자가 이것입니다.
이보다 훨씬 높게 시작하면 초기화가 잘못됐다는 신호예요. 직접 확인해볼까요?

In [ ]:
class GPTNoInit(GPT):
    def _init_weights(self, module):
        pass                                  # PyTorch 기본 초기값 그대로

bad = GPTNoInit(cfg)
bad_logits = bad(idx)
bad_loss = F.cross_entropy(bad_logits.view(B * T, -1), targets.view(B * T))

print(f"{'':16s} {'logits 표준편차':>14s} {'Loss':>8s}")
print(f"{'std 0.02 초기화':16s} {logits.std().item():14.3f} {loss.item():8.3f}")
print(f"{'기본 초기값':16s} {bad_logits.std().item():14.3f} {bad_loss.item():8.3f}")
print(f"{'기준 log(65)':16s} {'':14s} {math.log(65):8.3f}")

기본 초기값에서는 `wte`(= `lm_head`, Weight Tying)의 값이 커서 logits가 크게 튀고, 학습 전부터 **틀린 답을 자신 있게** 내서 Loss가 높습니다.
std 0.02로 작게 시작하면 logits가 0 근처라 Softmax가 거의 균등해집니다. (Day 5: 점수 차이가 작으면 확률이 고르게 퍼진다)

## 7. 파라미터 세기: GPT-2 small은 왜 1.24억 개인가?

### 7-1. 작은 모델 분해

In [ ]:
def breakdown(model):
    c = model.config
    blk = model.blocks[0]
    return {
        "wte (Token Emb, lm_head와 공유)": count(model.wte),
        "wpe (Position Emb)": count(model.wpe),
        f"Attention × {c.n_layer}": count(blk.attn) * c.n_layer,
        f"MLP × {c.n_layer}": count(blk.mlp) * c.n_layer,
        f"LayerNorm (Block 안) × {c.n_layer}": (count(blk.ln1) + count(blk.ln2)) * c.n_layer,
        "ln_f": count(model.ln_f),
        "lm_head (공유라 추가 0)": 0,
    }

parts = breakdown(model)
for k, v in parts.items():
    print(f"{k:34s} {v:>10,}")
print(f"{'합계':34s} {sum(parts.values()):>10,}")
print(f"{'count(model)':34s} {count(model):>10,}")

### 7-2. GPT-2 small 크기로 만들어 보기

GPT-2 small 크기를 그대로 만들면 메모리를 수백 MB 씁니다. 숫자만 세면 되니까 **`meta` 장치**에 만듭니다.
`meta`는 "shape만 있고 실제 값은 없는 텐서"라서 1초도 안 걸리고 메모리도 거의 안 써요.

In [ ]:
gpt2_cfg = GPTConfig(vocab_size=50257, block_size=1024, n_layer=12, n_head=12, n_embd=768)

with torch.device("meta"):
    gpt2 = GPT(gpt2_cfg)

parts2 = breakdown(gpt2)
total2 = sum(parts2.values())
for k, v in parts2.items():
    print(f"{k:34s} {v:>13,}  ({v / total2 * 100:5.1f}%)")
print(f"{'합계':34s} {total2:>13,}  → 약 {total2 / 1e6:.1f}M")

> 공식 GPT-2 small은 **124,439,808개**입니다. 우리 숫자와 몇만 개 차이가 나는 건 Day 14 버전 `c_attn`에 bias가 없어서예요. (12층 × 2304 = 27,648개)

In [ ]:
labels = [k.split(" (")[0] for k in parts2 if parts2[k] > 0]
values = [v / 1e6 for v in parts2.values() if v > 0]

plt.figure(figsize=(9, 3.8))
bars = plt.barh(labels[::-1], values[::-1], color="#4C72B0")
for b_, v in zip(bars, values[::-1]):
    plt.text(b_.get_width() + 0.5, b_.get_y() + b_.get_height() / 2, f"{v:.2f}M", va="center", fontsize=10)
plt.title("GPT-2 small 파라미터는 어디에 있나 (단위: 백만)", fontsize=13)
plt.xlim(0, max(values) * 1.18)
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

**관찰 포인트**
- **MLP가 가장 큽니다.** `C → 4C → C`라서 Block 하나에 약 `8C²`개. Attention(`약 4C²`)의 두 배예요.
- **wte도 엄청 큽니다.** 50257 × 768 ≈ 3,860만 개. Weight Tying이 없었다면 `lm_head`로 같은 양이 한 번 더 들어갔을 거예요.
- **LayerNorm은 거의 0입니다.** `C`마다 2개(scale, shift)뿐이에요.

### 7-3. Block 하나의 파라미터 공식

```text
LayerNorm × 2        : 2 × 2C          =  4C
Attention c_attn     : C × 3C          =  3C²        (bias 없음)
Attention proj       : C × C + C       =  C² + C
MLP Linear(C, 4C)    : C × 4C + 4C     =  4C² + 4C
MLP Linear(4C, C)    : 4C × C + C      =  4C² + C
─────────────────────────────────────────────────
합계                                   =  12C² + 10C
```

`C = 768`이면 `12 × 768² ≈ 708만`. 이게 12층이라 약 8,500만 개입니다.

In [ ]:
C = 768
formula = 12 * C**2 + 10 * C
print(f"공식       : {formula:,}")
print(f"실제 Block : {count(gpt2.blocks[0]):,}")

### 7-4. GPT-2 가족

설정 숫자만 바꾸면 같은 코드로 전부 만들 수 있습니다.

In [ ]:
family = {
    "GPT-2 small":  dict(n_layer=12, n_head=12, n_embd=768),
    "GPT-2 medium": dict(n_layer=24, n_head=16, n_embd=1024),
    "GPT-2 large":  dict(n_layer=36, n_head=20, n_embd=1280),
    "GPT-2 XL":     dict(n_layer=48, n_head=25, n_embd=1600),
}

print(f"{'모델':14s} {'n_layer':>8s} {'n_head':>7s} {'n_embd':>7s} {'파라미터':>10s}")
for name, kw in family.items():
    with torch.device("meta"):
        m = GPT(GPTConfig(vocab_size=50257, block_size=1024, **kw))
    print(f"{name:14s} {kw['n_layer']:8d} {kw['n_head']:7d} {kw['n_embd']:7d} {count(m) / 1e6:9.0f}M")

공식 발표 크기는 124M / 355M / 774M / 1.5B 입니다. 우리 코드로도 거의 같은 숫자가 나와요.
**구조는 완전히 같고, 숫자 몇 개만 다릅니다.** 요즘 LLM도 기본 뼈대는 크게 다르지 않아요.

## 8. LLM과의 연결

오늘 만든 `GPT` 클래스는 4주차 mini GPT의 **뼈대 그 자체**입니다. 남은 것은 두 가지예요.

| 남은 것 | 하는 일 | 언제 |
|---|---|---|
| 학습 데이터 만들기 + Loss | 입력 `x`와 한 칸 밀린 정답 `y`로 Cross Entropy | Day 20, 23 |
| 생성 루프 | 마지막 위치 logits → Softmax → Sampling → 이어붙이기 → 반복 | Day 20, 28 |

nanoGPT의 `model.py`를 열어보면 오늘 코드와 거의 같은 구조를 볼 수 있어요. (`wte`, `wpe`, `h`(= blocks), `ln_f`, `lm_head`, weight tying, std 0.02 초기화)

## 9. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `model`의 부품만 써서 GPT의 forward를 직접 완성하세요.
순서: Token Embedding + Position Embedding → Block들 → `ln_f` → `lm_head`

In [ ]:
def my_forward(model, idx):
    B, T = idx.shape
    # TODO: logits를 계산해서 return
    return None

test_idx = torch.randint(0, cfg.vocab_size, (2, 6))
out = my_forward(model, test_idx)
assert out is not None, "함수를 완성해주세요"
assert torch.allclose(out, model(test_idx), atol=1e-6), "순서나 빠진 부품이 없는지 확인해보세요"
print("Q1 정답!", tuple(out.shape))

**Q2.** 아래 설정의 모델에 `(B=3, T=5)` 입력을 넣었을 때 logits의 shape를 실행 **전에** 예측하세요.

In [ ]:
q2_cfg = GPTConfig(vocab_size=100, block_size=16, n_layer=3, n_head=2, n_embd=48)
q2_idx = torch.randint(0, 100, (3, 5))

my_shape = "?"   # TODO: 튜플로

assert my_shape == tuple(GPT(q2_cfg)(q2_idx).shape), "logits의 마지막 축이 무엇인지 생각해보세요"
print("Q2 정답!")

**Q3.** `vocab_size=1000`, `block_size=64`, `n_embd=128`인 GPT에서 **`wte` + `wpe` + `lm_head`** 의 파라미터 수를 손으로 계산하세요.
Weight Tying이 적용되어 있다는 것을 잊지 마세요.

In [ ]:
q3_cfg = GPTConfig(vocab_size=1000, block_size=64, n_layer=1, n_head=4, n_embd=128)
q3 = GPT(q3_cfg)

my_count = "?"   # TODO: 숫자 하나

shared = {id(p): p for m in (q3.wte, q3.wpe, q3.lm_head) for p in m.parameters()}
assert my_count == sum(p.numel() for p in shared.values()), "Weight Tying이면 lm_head가 따로 갖는 파라미터는?"
print("Q3 정답!")

**Q4.** `n_embd = 64`인 `Block` 하나의 파라미터 수를 손으로 계산하세요. (7-3의 표를 참고하세요)

In [ ]:
my_block_params = "?"   # TODO: 숫자 하나

blk = Block(n_embd=64, n_head=4, block_size=8)
assert my_block_params == count(blk), "7-3 표의 항목을 하나씩 더해보세요 (bias와 LayerNorm도 포함)"
print("Q4 정답!")

## 10. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. GPT는 Transformer와 어떤 관계인가? 왜 "Decoder-only"라고 부르는가?
2. GPT의 forward를 부품 이름과 shape로 순서대로 설명해보세요.
3. Pre-LN 구조에서 마지막에 `ln_f`를 한 번 더 넣는 이유는?
4. Weight Tying은 무엇을 공유하는가? 왜 shape가 딱 맞는가?
5. 학습 전 GPT의 Loss가 `log(vocab_size)` 근처인 이유는?
6. 입력이 `block_size`보다 길면 왜 안 되는가? 그럴 땐 어떻게 하는가?

**내 답:**

1.
2.
3.
4.
5.
6.